# 05 — Raster Alignment and Quality Control

Aligns all rasters to a selected reference raster and verifies CRS, shape,
resolution, transform and bounds.

In [ ]:
from pathlib import Path
import sys
import warnings

warnings.filterwarnings("ignore")

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").exists() and (candidate / "configs").exists():
            return candidate
    raise FileNotFoundError(
        "Project root was not found. Run this notebook from inside the repository."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
CONFIG_DIR = PROJECT_ROOT / "configs"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
MODEL_DIR = PROJECT_ROOT / "models"

for folder in [INTERIM_DIR, PROCESSED_DIR, OUTPUT_DIR, MODEL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import reproject, Resampling

predictor_files = sorted((RAW_DIR / "predictors" / "NDVI").glob("*.tif"))
if not predictor_files:
    predictor_files = sorted((RAW_DIR / "predictors").rglob("*.tif"))
if not predictor_files:
    raise FileNotFoundError("No reference raster could be found.")

reference_path = predictor_files[0]
print(f"Reference raster: {reference_path}")

In [ ]:
aligned_root = PROCESSED_DIR / "aligned_rasters"
aligned_root.mkdir(parents=True, exist_ok=True)

def align_raster(source_path: Path, output_path: Path, reference_path: Path) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with rasterio.open(reference_path) as ref, rasterio.open(source_path) as src:
        profile = ref.profile.copy()
        profile.update(
            count=1,
            dtype="float32",
            nodata=-9999.0,
            compress="lzw",
        )

        destination = np.full(
            (ref.height, ref.width),
            profile["nodata"],
            dtype="float32",
        )

        source_name = source_path.parent.name.lower()
        method = (
            Resampling.nearest
            if source_name in {"aspect"}
            else Resampling.bilinear
        )

        reproject(
            source=rasterio.band(src, 1),
            destination=destination,
            src_transform=src.transform,
            src_crs=src.crs,
            src_nodata=src.nodata,
            dst_transform=ref.transform,
            dst_crs=ref.crs,
            dst_nodata=profile["nodata"],
            resampling=method,
        )

        with rasterio.open(output_path, "w", **profile) as dst:
            dst.write(destination, 1)

all_rasters = sorted([
    *(RAW_DIR / "precipitation").rglob("*.tif"),
    *(RAW_DIR / "predictors").rglob("*.tif"),
])

for source in all_rasters:
    relative = source.relative_to(RAW_DIR)
    output = aligned_root / relative
    align_raster(source, output, reference_path)

print(f"Aligned {len(all_rasters)} rasters.")

In [ ]:
records = []
with rasterio.open(reference_path) as ref:
    for path in sorted(aligned_root.rglob("*.tif")):
        with rasterio.open(path) as src:
            records.append({
                "file": str(path.relative_to(PROJECT_ROOT)),
                "crs_match": src.crs == ref.crs,
                "shape_match": (src.height, src.width) == (ref.height, ref.width),
                "resolution_match": src.res == ref.res,
                "transform_match": src.transform == ref.transform,
                "bounds_match": src.bounds == ref.bounds,
            })

qc = pd.DataFrame(records)
display(qc)
qc.to_csv(PROCESSED_DIR / "raster_alignment_qc.csv", index=False)